## Initial EDA and Data Cleaning

In [21]:
import pandas as pd
import numpy as np
import os
import glob  # Built-in module for finding files matching patterns (e.g., *.csv)
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


In [22]:
# Load customer data(using relative path)
customers_df = pd.read_csv('data/customers.csv', delimiter='|')
print(f"Customers dataset shape: {customers_df.shape}")
print(f"Customers columns: {list(customers_df.columns)}")


Customers dataset shape: (1010, 16)
Customers columns: ['ssn', 'cc_num', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'acct_num', 'profile']


In [23]:
customers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ssn       1010 non-null   object 
 1   cc_num    1010 non-null   int64  
 2   first     1010 non-null   object 
 3   last      1010 non-null   object 
 4   gender    1010 non-null   object 
 5   street    1010 non-null   object 
 6   city      1010 non-null   object 
 7   state     1010 non-null   object 
 8   zip       1010 non-null   int64  
 9   lat       1010 non-null   float64
 10  long      1010 non-null   float64
 11  city_pop  1010 non-null   int64  
 12  job       1010 non-null   object 
 13  dob       1010 non-null   object 
 14  acct_num  1010 non-null   int64  
 15  profile   1010 non-null   object 
dtypes: float64(2), int64(4), object(10)
memory usage: 126.4+ KB


##### The dataset was generated using the Sparkov Data Generation tool. There are not null values in the dataset.

In [24]:
customers_df.head()

,ssn,cc_num,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,acct_num,profile
0,115-04-4507,4218196001337,Kathy,Johnson,F,863 Lawrence Valleys,Staten Island,NY,10302,40.6306,-74.1379,468730,Accounting technician,1982-10-03,888022315787,adults_2550_female_urban.json
1,715-55-5575,4351161559407816183,Elaine,Fuller,F,310 Kendra Common Apt. 164,Peabody,MA,1960,42.5326,-70.9612,50944,Professor Emeritus,1994-06-07,917558277935,adults_2550_female_urban.json
2,167-48-5821,4192832764832,Melinda,Cameron,F,05641 Robin Port,Waukomis,OK,73773,36.2781,-97.8996,1744,International aid/development worker,1934-05-30,718172762479,adults_50up_female_rural.json
3,406-83-7518,4238849696532874,Brandon,Williams,M,26916 Carlson Mountain,Los Angeles,CA,90019,34.0482,-118.3343,2383912,Seismic interpreter,1991-12-26,947268892251,adults_2550_male_urban.json
4,697-93-1877,4514627048281480,Lisa,Hernandez,F,809 Burns Creek,Austin,TX,78727,30.4254,-97.7195,940359,Medical laboratory scientific officer,1998-05-22,888335239225,adults_2550_female_urban.json


In [25]:
# Transform a few columns to string because we won't be doing any math with them
columns_to_transform = ['cc_num', 'zip', 'acct_num']
customers_df[columns_to_transform] = customers_df[columns_to_transform].astype(str)
customers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ssn       1010 non-null   object 
 1   cc_num    1010 non-null   object 
 2   first     1010 non-null   object 
 3   last      1010 non-null   object 
 4   gender    1010 non-null   object 
 5   street    1010 non-null   object 
 6   city      1010 non-null   object 
 7   state     1010 non-null   object 
 8   zip       1010 non-null   object 
 9   lat       1010 non-null   float64
 10  long      1010 non-null   float64
 11  city_pop  1010 non-null   int64  
 12  job       1010 non-null   object 
 13  dob       1010 non-null   object 
 14  acct_num  1010 non-null   object 
 15  profile   1010 non-null   object 
dtypes: float64(2), int64(1), object(13)
memory usage: 126.4+ KB


In [31]:
# The profile column contains the customer's demographic profile. 
# It has the format:
# "young_adults_male_urban.json"  or "adults_50up_female_rural.json".
# Split from the right to get the last 3 parts consistently
profile_parts = customers_df['profile'].str.rsplit('_', n=2, expand=True)

# Now the columns are more predictable:
# Column 0: everything before the last 3 underscores (pop_group)
# Column 1: age_group  
# Column 2: gender
# Column 3: location.json

customers_df['pop_group'] = profile_parts[0]
customers_df['age_group'] = profile_parts[1] 
customers_df['location'] = profile_parts[3].str.split('.').str[0] # Remove .json extension

# customers_df['gender'] = profile_parts[2]  # Skipping gender since is not needed(for now)
customers_df['location'] = profile_parts[3].str.split('.').str[0]  
# Drop the original profile column
#customers_df.drop(columns=['profile'], inplace=True)
customers_df.head()

KeyError: 3

In [ ]:
print(f"Population groups: {customers_df['pop_group'].unique()}")
print(f"Age groups: {customers_df['age_group'].unique()}")
print(f"Locations: {customers_df['location'].unique()}")

Age groups: ['2550' '50up' 'adults']
Population groups: ['adults' 'young']
Locations: ['urban' 'rural']


In [29]:
customers_df[customers_df['age_group'] == 'adults'].value_counts()

ssn          cc_num            first        last        gender  street                            city         state  zip    lat      long       city_pop  job                                          dob         acct_num      profile                         pop_group  age_group  location
004-15-4566  676305278100      Timothy      Richardson  M       02722 Kyle Island                 Pleasanton   TX     78064  28.9924  -98.4831   14102     Psychologist, educational                    2005-12-04  930558644123  young_adults_male_urban.json    young      adults     urban       1
006-01-0857  6558682405166545  Erika        Moore       F       4189 Fitzpatrick Avenue Apt. 338  Irwin        PA     15642  40.3191  -79.7205   45286     Optometrist                                  2001-05-05  588033203866  young_adults_female_urban.json  young      adults     urban       1
013-37-5652  377813767426621   Russell      Harper      M       47851 Davis Lodge                 New Orleans  LA     70115

In [20]:
print("Age group distribution:\n", customers_df['age_group'].value_counts().reset_index())

print("Population group distribution:\n", customers_df['pop_group'].value_counts().reset_index())

print("Location distribution:\n", customers_df['location'].value_counts().reset_index())

Age group distribution:
   age_group  count
0      50up    516
1      2550    401
2    adults     93
Population group distribution:
   pop_group  count
0    adults    917
1     young     93
Location distribution:
   location  count
0    urban    969
1    rural     41
